In [ ]:
import re, unicodedata
from rank_bm25 import BM25Okapi   # pip install rank-bm25

STOPWORDS = {"le","la","les","de","des","du","un","une","et","ou","en","à","a",
             "au","aux","pour","par","sur","dans","est","que","qui","ce","se","l","d"}

def tokeniser(texte: str) -> list[str]:
    texte = unicodedata.normalize("NFD", texte.lower())
    texte = "".join(c for c in texte if unicodedata.category(c) != "Mn")  # enlève les accents
    return [t for t in re.findall(r"\w+", texte) if t not in STOPWORDS]

In [2]:
"degats"==tokeniser("Dégâts")

False

In [3]:
tokeniser("Dégâts")

['degats']

In [5]:
# version 2
from nltk.stem.snowball import FrenchStemmer     # pip install nltk (pas de téléchargement nécessaire)
stem = FrenchStemmer().stem

STOPWORDS |= {"suis", "me", "moi", "mon", "etre", "ai", "est-ce"}

def tokeniser(texte):
    texte = texte.lower().replace("ʼ", "'").replace("’", "'")
    texte = unicodedata.normalize("NFD", texte)
    texte = "".join(c for c in texte if unicodedata.category(c) != "Mn")
    return [stem(t) for t in re.findall(r"\w+", texte) if len(t) > 1 and t not in STOPWORDS]

# print(tokeniser(question))                     # attendu : 'vol', 'cav'... ("vole" et "vol" -> même racine)
# bm25 = BM25Okapi([tokeniser(t) for t in data["documents"]])   # on reconstruit l'index avec le nouveau tokeniseur

In [6]:
def construire_index_bm25(chunks):          # chunks = liste de dicts {id, texte, contrat_id, page...}
    index = {}
    for cid in {c["contrat_id"] for c in chunks}:
        docs = [c for c in chunks if c["contrat_id"] == cid]
        index[cid] = (BM25Okapi([tokeniser(c["texte"]) for c in docs]), docs)
    return index

def rechercher_bm25(index, question, contrat_id, k=20):
    bm25, docs = index[contrat_id]
    scores = bm25.get_scores(tokeniser(question))
    meilleurs = sorted(range(len(docs)), key=lambda i: scores[i], reverse=True)[:k]
    return [docs[i] for i in meilleurs]

In [ ]:
import sys, os
from pathlib import Path

RACINE = Path.cwd().parents[1]          # src/hybride -> racine du projet
os.chdir(RACINE); sys.path.insert(0, str(RACINE))

from src.retrieval.retrieval import ouvrir_store, rechercher
from src.extraction_cp.extraction import analyser_cp

store = ouvrir_store()
_, _, contrat_id = analyser_cp(Path("data/cp/cp_01_luko_etendue.pdf"))
print("Contrat :", contrat_id)

Contrat : luko_hce_2504


In [8]:
data = store.get(where={"contrat_id": contrat_id}, include=["documents", "metadatas"])
print(len(data["documents"]), "chunks pour ce contrat")
print(data["metadatas"][0])
print(data["documents"][0][:300])

359 chunks pour ce contrat
{'reference': 'HCE', 'contrat_id': 'luko_hce_2504', 'produit': 'Habitation', 'type_doc': 'CG', 'fichier': 'luko_habitation-etendue_cg_2504.pdf', 'chunk_id': 'luko_habitation-etendue_cg_2504.pdf#0', 'alias': 'Luko by Allianz Direct, Allianz Direct', 'page': 2, 'edition': '2504', 'assureur': 'Luko', 'formule': 'Couverture Étendue'}
Note à nos lecteurs 

On a voulu vous écrire le plus beau contrat 
d'assurance habitation du monde. 

Avant d'aller dans le détail de ces conditions générales, répondons d'abord à la question à 1 million 
d'euros : à quoi ça sert ? d'où ça sort exactement ?


In [9]:
print(tokeniser("Suis-je couvert si on me vole mon vélo dans ma cave ?"))
print(tokeniser("Les dégâts des eaux et l’assurance"))

['je', 'couvert', 'si', 'on', 'vol', 'velo', 'ma', 'cav']
['degat', 'eau', 'assur']


In [12]:
from rank_bm25 import BM25Okapi     # pip install rank-bm25

bm25 = BM25Okapi([tokeniser(t) for t in data["documents"]])

question = "Dans quel délai dois-je déclarer un vol à mon assureur ?"
scores = bm25.get_scores(tokeniser(question))
top_bm25 = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]

print("=== BM25")
for i in top_bm25:
    m = data["metadatas"][i]
    print(f"{scores[i]:5.1f}  {m['fichier']} p.{m['page']}  {data['documents'][i][:90]!r}")

print("\n=== Vecteurs")
for p in rechercher(store, question, contrat_id, k=5):
    print(f"       {p.metadata['fichier']} p.{p.metadata['page']}  {p.page_content[:90]!r}")

=== BM25
 11.6  luko_habitation-etendue_cg_2504.pdf p.67  'VI. Quelles démarches en cas de sinistre ? \n\n1. Les délais à respecter \nVous (ou vos ayant'
 10.9  luko_habitation-etendue_ipid_2504.pdf p.2  'Où suis-je couvert(e) ? Pour la garantie dommage aux biens : \n✔\n\nEn\n\nFrance\n\nmétropolitain'
  8.8  luko_habitation-etendue_cg_2504.pdf p.92  "5. Obligation de l'Assuré \nL'assuré doit déclarer à l'assureur ou à son représentant local"
  7.6  luko_habitation-etendue_cg_2504.pdf p.3  'Une assurance habitation ne vous couvre pas en revanche quand vos biens sont usés à \ncause'
  7.2  luko_habitation-etendue_cg_2504.pdf p.88  "2.1. Comment fonctionne le mode de déclenchement par « le \nfait dommageable » ? \nL'assureu"

=== Vecteurs
       luko_habitation-etendue_cg_2504.pdf p.68  "Vous devez fournir un état des pertes, c'est-à-dire une estimation sincère et signée de vo"
       luko_habitation-etendue_cg_2504.pdf p.67  'VI. Quelles démarches en cas de sinistre ? \n\n1. Les délais 

In [13]:
def cle(m, texte):
    return (m["fichier"], m["page"], texte)

def fusion_rrf(listes, k=5, rrf_k=60):
    """Chaque liste = [(metadata, texte), ...] dans l'ordre. On additionne 1/(rrf_k + rang)."""
    scores, items = {}, {}
    for liste in listes:
        for rang, (m, t) in enumerate(liste, start=1):
            c = cle(m, t)
            scores[c] = scores.get(c, 0) + 1 / (rrf_k + rang)
            items[c] = (m, t)
    return [(items[c], scores[c]) for c in sorted(scores, key=scores.get, reverse=True)[:k]]

def liste_bm25(question, n=20):
    s = bm25.get_scores(tokeniser(question))
    return [(data["metadatas"][i], data["documents"][i]) for i in sorted(range(len(s)), key=lambda i: s[i], reverse=True)[:n]]

def liste_vect(question, n=20):
    return [(p.metadata, p.page_content) for p in rechercher(store, question, contrat_id, k=n)]

for (m, t), s in fusion_rrf([liste_vect(question), liste_bm25(question)]):
    print(f"{s:.4f}  p.{m['page']}  {t[:80]!r}")

0.0325  p.67  'VI. Quelles démarches en cas de sinistre ? \n\n1. Les délais à respecter \nVous (ou'
0.0317  p.92  "5. Obligation de l'Assuré \nL'assuré doit déclarer à l'assureur ou à son représen"
0.0286  p.2  'sécurité\n\ndu\n\nbien,\n\n-\n\nMettre\n\nen\n\nœ\nuvre\n\nles\n\nmoyens\n\nde\n\nprotection\n\net\n\nde\n'
0.0284  p.92  "6. Obligation de l'assureur \nL'assureur doit verser l'indemnité due au titre de "
0.0274  p.67  '2. Les formalités à accomplir \nVous devez faire une déclaration claire et précis'


In [20]:
import json, time

questions = json.load(open(Path("data/eval/questions.json"), encoding="utf-8"))   # <- ton fichier
index_par_contrat = {}

def outils(cid):
    """Index BM25 construit une fois par contrat (même filtre que les vecteurs)."""
    if cid not in index_par_contrat:
        d = store.get(where={"contrat_id": cid}, include=["documents", "metadatas"])
        index_par_contrat[cid] = (BM25Okapi([tokeniser(t) for t in d["documents"]]), d)
    return index_par_contrat[cid]

def top_k(methode, q, cid, k=5, n=20):
    bm, d = outils(cid)
    vect = [(p.metadata, p.page_content) for p in rechercher(store, q, cid, k=n)]
    s = bm.get_scores(tokeniser(q))
    lex = [(d["metadatas"][i], d["documents"][i]) for i in sorted(range(len(s)), key=lambda i: s[i], reverse=True)[:n]]
    if methode == "vecteurs": return vect[:k]
    if methode == "bm25":     return lex[:k]
    return [x for x, _ in fusion_rrf([vect, lex], k=k)]

contrats = {}
def contrat_de(cp):
    """analyser_cp une seule fois par CP (~5 s chacune)."""
    if cp not in contrats:
        contrats[cp] = analyser_cp(Path("data/cp") / cp)[2]
    return contrats[cp]

evaluables = [q for q in questions if q.get("sources")]
print(len(evaluables), "questions évaluables")      # attendu : 16 (comme la baseline)

for methode in ["vecteurs", "bm25", "hybride"]:
    hits, recalls, t0 = [], [], time.perf_counter()
    for item in evaluables:
        cid = contrat_de(item["cp"])
        attendues = {(s["fichier"], int(p)) for s in item["sources"] for p in s["pages"]}
        trouvees = {(m["fichier"], int(m["page"])) for m, _ in top_k(methode, item["question"], cid)}
        hits.append(bool(attendues & trouvees))
        
        recalls.append(len(attendues & trouvees) / len(attendues) if len(attendues) > 0 else 0)
    n = len(evaluables)
    print(f"{methode:9} Hit@5={sum(hits)/n:.0%}  Recall@5={sum(recalls)/n:.0%}  ({(time.perf_counter()-t0)/n:.2f} s/question)")

23 questions évaluables
vecteurs  Hit@5=57%  Recall@5=34%  (1.44 s/question)
bm25      Hit@5=48%  Recall@5=23%  (0.12 s/question)
hybride   Hit@5=57%  Recall@5=33%  (0.12 s/question)


In [18]:
print(type(questions), len(questions))
print(questions[0] if isinstance(questions, list) else list(questions.items())[0])

<class 'list'> 25
{'id': 'Q01', 'type': 'CG', 'cp': 'cp_01_luko_etendue.pdf', 'question': 'Dans quel délai dois-je déclarer un vol à mon assureur ?', 'reponse_attendue': "Dans les 2 jours ouvrés après en avoir eu connaissance, via l'application Luko. Il faut aussi prévenir immédiatement les autorités, envoyer l'original du dépôt de plainte, et fournir l'état des pertes sous 8 jours (à remettre aussi à la police ou à la gendarmerie). Si la déclaration est en retard et que ce retard cause un préjudice à l'assureur, l'assuré perd son droit à indemnité (déchéance).", 'sources': [{'fichier': 'luko_habitation-etendue_cg_2504.pdf', 'pages': [67]}]}


In [1]:
import sys, os, re, json, time, unicodedata
from pathlib import Path

RACINE = Path.cwd().parents[1]           # src/hybride -> racine du projet
os.chdir(RACINE)
sys.path.insert(0, str(RACINE))

from rank_bm25 import BM25Okapi
from nltk.stem.snowball import FrenchStemmer
from src.retrieval.retrieval import ouvrir_store, rechercher
from src.extraction_cp.extraction import analyser_cp

store = ouvrir_store()
questions = json.load(open("data/eval/questions.json", encoding="utf-8"))
print(store._collection.count(), "chunks |", len(questions), "questions")

2368 chunks | 25 questions


In [2]:
stem = FrenchStemmer().stem

STOPWORDS = {"le", "la", "les", "de", "des", "du", "un", "une", "et", "ou", "en", "a", "au", "aux",
             "pour", "par", "sur", "dans", "est", "que", "qui", "ce", "se", "l", "d",
             "suis", "me", "moi", "mon", "ma", "mes", "etre", "ai", "je", "il", "vous", "votre"}

def tokeniser(texte):
    """minuscules -> apostrophes normalisées -> sans accents -> mots -> sans mots vides -> racines"""
    texte = texte.lower().replace("ʼ", "'").replace("’", "'")
    texte = unicodedata.normalize("NFD", texte)
    texte = "".join(c for c in texte if unicodedata.category(c) != "Mn")
    return [stem(t) for t in re.findall(r"\w+", texte) if len(t) > 1 and t not in STOPWORDS]

print(tokeniser("Suis-je couvert si on me vole mon vélo dans ma cave ?"))

['couvert', 'si', 'on', 'vol', 'velo', 'cav']


In [3]:
contrats = {}
def contrat_de(nom_cp):
    """analyser_cp une seule fois par CP."""
    if nom_cp not in contrats:
        contrats[nom_cp] = analyser_cp(Path("data/cp") / nom_cp)[2]
    return contrats[nom_cp]

index_bm25 = {}
def outils(contrat_id):
    """Un index BM25 par contrat : le MÊME filtre que la recherche vectorielle."""
    if contrat_id not in index_bm25:
        d = store.get(where={"contrat_id": contrat_id}, include=["documents", "metadatas"])
        index_bm25[contrat_id] = (BM25Okapi([tokeniser(t) for t in d["documents"]]), d)
    return index_bm25[contrat_id]

contrat_id = contrat_de("cp_01_luko_etendue.pdf")
bm, d = outils(contrat_id)
print(contrat_id, "|", len(d["documents"]), "chunks dans l'index BM25 de ce contrat")

luko_hce_2504 | 359 chunks dans l'index BM25 de ce contrat


In [4]:
def fusion_rrf(listes, k=5, rrf_k=60):
    """Reciprocal Rank Fusion : chaque chunk gagne 1/(rrf_k + rang) dans chaque liste où il apparaît."""
    scores, items = {}, {}
    for liste in listes:
        for rang, (m, t) in enumerate(liste, start=1):
            cle = m["chunk_id"]
            scores[cle] = scores.get(cle, 0) + 1 / (rrf_k + rang)
            items[cle] = (m, t)
    return [items[c] for c in sorted(scores, key=scores.get, reverse=True)[:k]]

def recherche(methode, question, contrat_id, k=5, n=20):
    if methode == "vecteurs":
        return [(p.metadata, p.page_content) for p in rechercher(store, question, contrat_id, k=k)]
    bm, d = outils(contrat_id)
    s = bm.get_scores(tokeniser(question))
    lex = [(d["metadatas"][i], d["documents"][i])
           for i in sorted(range(len(s)), key=lambda i: s[i], reverse=True)[:n]]
    if methode == "bm25":
        return lex[:k]
    vect = [(p.metadata, p.page_content) for p in rechercher(store, question, contrat_id, k=n)]
    return fusion_rrf([vect, lex], k=k)

In [5]:
question = "Suis-je couvert si on me vole mon vélo dans ma cave ?"
for methode in ["vecteurs", "bm25", "hybride"]:
    print(f"=== {methode}")
    for m, t in recherche(methode, question, contrat_id):
        print(f"   p.{m['page']:<4} {t[:80]!r}")

=== vecteurs
   p.47   'Nous vous garantissons également contre les recours dirigés contre vous en \nrais'
   p.12   '3. Quels sont les biens assurés ?'
   p.23   '1.5. Le vol et le vandalisme \n Vol \nNous garantissons, sous réserve du niveau de'
   p.36   'Garanties supplémentaires \nNous couvrons également : \n● le vol des biens mobilie'
   p.24   'Garantie additionnelle \nNous couvrons aussi les dommages matériels des biens imm'
=== bm25
   p.26   "Biens couverts Plafonds d'indemnisation \nHabitation À concurrence des dommages \n"
   p.2    'Dans ce cadre, nous faisons attention à toujours employer le mot exact et à pren'
   p.3    'Une assurance habitation ne vous couvre pas en revanche quand vos biens sont usé'
   p.40   '3.3. Frais de déplacement du mobilier \nLes frais rendus indispensables à la suit'
   p.25   "En cas d'inhabitation de vos locaux, la garantie est suspendue dans les conditio"
=== hybride
   p.23   '1.5. Le vol et le vandalisme \n Vol \nNous garantissons, sous r

In [6]:
def pages_cg_attendues(q):
    """Seulement les pages des CG : la CP n'est jamais dans l'index."""
    return {(s["fichier"], p) for s in q["sources"] if not s["fichier"].startswith("cp_")
            for p in s["pages"] if isinstance(p, int)}

evaluables = [q for q in questions if pages_cg_attendues(q)]
print(len(evaluables), "questions évaluables")       # doit afficher 16

for q in evaluables:                                  # préparation HORS chronomètre
    outils(contrat_de(q["cp"]))

for methode in ["vecteurs", "bm25", "hybride"]:
    hits, recalls, t0 = [], [], time.perf_counter()
    for q in evaluables:
        attendues = pages_cg_attendues(q)
        trouvees = {(m["fichier"], m["page"]) for m, _ in recherche(methode, q["question"], contrat_de(q["cp"]))}
        hits.append(bool(attendues & trouvees))
        recalls.append(len(attendues & trouvees) / len(attendues))
    n = len(evaluables)
    print(f"{methode:9} Hit@5={sum(hits)/n:.0%}  Recall@5={sum(recalls)/n:.0%}  "
          f"({(time.perf_counter() - t0) / n:.2f} s/question)")

16 questions évaluables
vecteurs  Hit@5=81%  Recall@5=51%  (0.07 s/question)
bm25      Hit@5=75%  Recall@5=39%  (0.00 s/question)
hybride   Hit@5=81%  Recall@5=51%  (0.07 s/question)


In [7]:
import pandas as pd
lignes = []
for q in evaluables:
    attendues, cid = pages_cg_attendues(q), contrat_de(q["cp"])
    ligne = {"id": q["id"]}
    for methode in ["vecteurs", "hybride"]:
        trouvees = {(m["fichier"], m["page"]) for m, _ in recherche(methode, q["question"], cid)}
        ligne[methode] = round(len(attendues & trouvees) / len(attendues), 2)
    lignes.append(ligne)
comp = pd.DataFrame(lignes)
comp["ecart"] = comp["hybride"] - comp["vecteurs"]
comp[comp["ecart"] != 0]

,id,vecteurs,hybride,ecart


In [8]:
import torch
from sentence_transformers import CrossEncoder

appareil = "cuda" if torch.cuda.is_available() else "cpu"
print("Appareil :", appareil)

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device=appareil)
print(reranker.predict([("vol de vélo dans la cave", "Le vol commis dans les caves est garanti si le local est fermé à clé."),
                        ("vol de vélo dans la cave", "Vous pouvez résilier votre contrat chaque année.")]))

c:\Users\pc\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Appareil : cuda


Loading weights: 100%|██████████| 393/393 [00:02<00:00, 161.39it/s]


[1.9826593e-01 1.6322725e-05]


In [9]:
def recherche_rerank(question, contrat_id, base="hybride", n=20, k=5):
    # 1. 20 candidats
    if base == "hybride":
        bm, d = outils(contrat_id)
        s = bm.get_scores(tokeniser(question))
        lex = [(d["metadatas"][i], d["documents"][i]) for i in sorted(range(len(s)), key=lambda i: s[i], reverse=True)[:n]]
        vect = [(p.metadata, p.page_content) for p in rechercher(store, question, contrat_id, k=n)]
        candidats = fusion_rrf([vect, lex], k=n)
    else:
        candidats = [(p.metadata, p.page_content) for p in rechercher(store, question, contrat_id, k=n)]

    # 2. Le cross-encoder note chaque paire (question, chunk)
    scores = reranker.predict([(question, t) for _, t in candidats])
    classes = [c for _, c in sorted(zip(scores, candidats), key=lambda x: x[0], reverse=True)]

    # 3. Un seul chunk par page, pour ne pas gaspiller une place
    resultat, pages_vues = [], set()
    for m, t in classes:
        if (m["fichier"], m["page"]) not in pages_vues:
            pages_vues.add((m["fichier"], m["page"]))
            resultat.append((m, t))
        if len(resultat) == k:
            break
    return resultat

In [10]:
for base in ["vecteurs", "hybride"]:
    print(f"=== {base} + reranking")
    for m, t in recherche_rerank(question, contrat_id, base=base):
        print(f"   p.{m['page']:<4} {t[:80]!r}")

=== vecteurs + reranking
   p.47   'Nous vous garantissons également contre les recours dirigés contre vous en \nrais'
   p.23   '1.5. Le vol et le vandalisme \n Vol \nNous garantissons, sous réserve du niveau de'
   p.25   "En cas d'inhabitation de vos locaux, la garantie est suspendue dans les conditio"
   p.24   'Garantie additionnelle \nNous couvrons aussi les dommages matériels des biens imm'
   p.36   'Garanties supplémentaires \nNous couvrons également : \n● le vol des biens mobilie'
=== hybride + reranking
   p.47   'Nous vous garantissons également contre les recours dirigés contre vous en \nrais'
   p.23   '1.5. Le vol et le vandalisme \n Vol \nNous garantissons, sous réserve du niveau de'
   p.25   "En cas d'inhabitation de vos locaux, la garantie est suspendue dans les conditio"
   p.24   'Garantie additionnelle \nNous couvrons aussi les dommages matériels des biens imm'
   p.26   "Biens couverts Plafonds d'indemnisation \nHabitation À concurrence des dommages \n"


In [11]:
def evaluer(nom, fonction):
    hits, recalls, t0 = [], [], time.perf_counter()
    for q in evaluables:
        attendues = pages_cg_attendues(q)
        trouvees = {(m["fichier"], m["page"]) for m, _ in fonction(q["question"], contrat_de(q["cp"]))}
        hits.append(bool(attendues & trouvees))
        recalls.append(len(attendues & trouvees) / len(attendues))
    n = len(evaluables)
    print(f"{nom:20} Hit@5={sum(hits)/n:.0%}  Recall@5={sum(recalls)/n:.0%}  ({(time.perf_counter() - t0)/n:.2f} s/question)")

evaluer("vecteurs", lambda q, c: recherche("vecteurs", q, c))
evaluer("vecteurs + rerank", lambda q, c: recherche_rerank(q, c, base="vecteurs"))
evaluer("hybride + rerank", lambda q, c: recherche_rerank(q, c, base="hybride"))

vecteurs             Hit@5=81%  Recall@5=51%  (0.08 s/question)
vecteurs + rerank    Hit@5=88%  Recall@5=50%  (0.66 s/question)
hybride + rerank     Hit@5=81%  Recall@5=51%  (0.65 s/question)


In [12]:
import time
from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama
from src.generation.generation import PROMPT, formater_passages, ajouter_sources

passages = rechercher(store, question, contrat_id)          # les mêmes 5 passages pour les deux modèles
prompt = PROMPT.format(cp=texte_cp, passages=formater_passages(passages), question=question)

modeles = {
    "qwen3:8b (local)": ChatOllama(model="qwen3:8b", temperature=0, num_ctx=8192, num_predict=600, reasoning=False),
    "gpt-oss-120b (Groq)": ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_tokens=1500),
}
for nom, llm in modeles.items():
    debut = time.time()
    rep = llm.invoke(prompt)
    print(f"===== {nom} ({time.time() - debut:.1f} s)")
    print(ajouter_sources(rep.content, passages), "\n")

NameError: name 'texte_cp' is not defined